# Newton's Method and Newton--Raphson

**S&DS 432/632 -- Advanced Optimization Techniques**

This notebook accompanies Chapter 4 (Newton Methods) of the lecture notes. It contains:

1. **Newton's method for minimization** -- quadratic approximation demos
2. **Newton--Raphson for root-finding** -- tangent-line iteration
3. **Convergence behaviors** -- sensitivity to initialization, cycling, divergence
4. **Newton fractals** -- chaotic dependence on initialization in the complex plane
5. **Damped Newton and line search** -- remedies for divergence
6. **Convergence experiments** -- Phase I/II behavior, comparison with gradient descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. Newton's Method for Minimization

Newton's method minimizes $f(x)$ by iteratively fitting a **quadratic model** at the current point and jumping to its minimizer:

$$x_{k+1} = x_k - \frac{f'(x_k)}{f''(x_k)}$$

### Example: Minimizing $f(x) = x \log(x) - 2x$

This is a convex function on $x > 0$ with minimizer at $x^* = e$.

In [ ]:
def f_min(x):
    return x * np.log(x) - 2 * x

def df_min(x):
    return np.log(x) - 1

def ddf_min(x):
    return 1 / x

def taylor_quadratic(x, x0):
    """Second-order Taylor expansion of f at x0."""
    return f_min(x0) + df_min(x0) * (x - x0) + 0.5 * ddf_min(x0) * (x - x0)**2

# Newton iterates starting from x0 = 6
x0 = 6.0
iterates = [x0]
for _ in range(2):
    x0 = x0 - df_min(x0) / ddf_min(x0)
    iterates.append(x0)

# Plot
x = np.linspace(0.3, 7, 400)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, f_min(x), 'k-', linewidth=2.5, label='$f(x) = x\\log(x) - 2x$')

colors = ['#C47A6A', '#7A9A7E', '#7B97AD']
for i, xi in enumerate(iterates):
    y_approx = taylor_quadratic(x, xi)
    ax.plot(x, y_approx, '--', color=colors[i], linewidth=1.8,
            label=f'Quadratic model at $x = {xi:.3f}$')
    ax.plot(xi, f_min(xi), 'o', color=colors[i], markersize=8)

ax.axvline(np.e, color='gray', linestyle=':', alpha=0.5, label=f'$x^* = e \\approx {np.e:.3f}$')
ax.set_ylim(-4, 2)
ax.set_xlabel('$x$')
ax.set_ylabel('$f(x)$')
ax.set_title("Newton's Method: Quadratic Approximations")
ax.legend()
plt.tight_layout()
plt.show()

print('Iterates:', [f'{xi:.6f}' for xi in iterates])
print(f'True minimizer: e = {np.e:.6f}')

## 2. Newton--Raphson for Root-Finding

Newton--Raphson solves $F(x) = 0$ by linearizing at the current point:

$$x_{k+1} = x_k - \frac{F(x_k)}{F'(x_k)}$$

Geometrically, $x_{k+1}$ is where the **tangent line** at $(x_k, F(x_k))$ crosses zero.

### Example: Finding roots of $F(x) = \log(x) - 1$

In [ ]:
F = lambda x: np.log(x) - 1
dF = lambda x: 1 / x

x0 = 6.0
x_plot = np.linspace(0.1, 7, 400)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_plot, F(x_plot), 'k-', linewidth=2.5, label='$F(x) = \\log(x) - 1$')
ax.axhline(0, color='black', lw=0.5)

colors = ['#C47A6A', '#7A9A7E', '#7B97AD']
for i in range(3):
    m = dF(x0)
    tangent = lambda x, x0=x0, m=m: F(x0) + m * (x - x0)
    ax.plot(x_plot, tangent(x_plot), '--', color=colors[i], linewidth=1.8,
            label=f'Tangent at $x = {x0:.3f}$')
    ax.plot(x0, F(x0), 'o', color=colors[i], markersize=8)
    x0 = x0 - F(x0) / dF(x0)

ax.axvline(np.e, color='gray', linestyle=':', alpha=0.5, label=f'Root: $x^* = e$')
ax.set_ylim(-2, 3)
ax.set_xlabel('$x$')
ax.set_ylabel('$F(x)$')
ax.set_title('Newton--Raphson: Tangent Line Iterations')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Convergence Behaviors

Newton--Raphson does **not always converge**. Its behavior is sensitive to initialization and the structure of $F$.

### 3.1 Sensitivity to initialization: $F(x) = x^2 - 9$

The two roots are $x = \pm 3$. Starting from $x_0 > 0$ converges to $+3$; from $x_0 < 0$ to $-3$; from $x_0 = 0$ the method fails ($F'(0) = 0$).

In [ ]:
F_sq = lambda x: x**2 - 9
dF_sq = lambda x: 2 * x

def newton_raphson(F, dF, x0, n_iter=10):
    iterates = [x0]
    for _ in range(n_iter):
        x0 = x0 - F(x0) / dF(x0)
        iterates.append(x0)
    return iterates

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
x_plot = np.linspace(-6, 6, 400)

for ax, x0 in zip(axes, [5, 1, -1, -5]):
    iterates = newton_raphson(F_sq, dF_sq, x0, n_iter=5)
    ax.plot(x_plot, F_sq(x_plot), 'k-', linewidth=1.5)
    ax.axhline(0, color='black', lw=0.3)
    for i, xi in enumerate(iterates[:5]):
        ax.plot(xi, 0, 'o', color=plt.cm.viridis(i / 5), markersize=7)
    ax.set_title(f'$x_0 = {x0}$', fontsize=11)
    ax.set_xlabel('$x$')
    ax.set_ylim(-12, 20)

axes[0].set_ylabel('$F(x) = x^2 - 9$')
fig.suptitle('Convergence depends on initialization', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Chaotic sensitivity: $F(x) = x^3 - 2x^2 - 11x + 12$

This polynomial has three roots: $x = -3, 1, 4$. Five initial points that differ by less than $10^{-4}$ converge to **three different roots**.

In [ ]:
import pandas as pd

F_poly = lambda x: x**3 - 2*x**2 - 11*x + 12
dF_poly = lambda x: 3*x**2 - 4*x - 11

initial_points = [2.352875270, 2.352841720, 2.352837350, 2.352836327, 2.352836323]

trajectories = []
for x0 in initial_points:
    iterates = newton_raphson(F_poly, dF_poly, x0, n_iter=40)
    trajectories.append(iterates)

df = pd.DataFrame(trajectories).T
df.columns = [f'x0 = {x0:.9f}' for x0 in initial_points]
print('Last iterate for each starting point:')
print(df.iloc[-1].to_string())
print()
print('First 10 iterates:')
print(df.head(10).to_string())

### 3.3 Cycling: $F(x) = x^3 - 2x + 2$, $x_0 = 0$

With $F'(x) = 3x^2 - 2$, the update gives $x_1 = 1$, $x_2 = 0$, $x_3 = 1, \ldots$ --- a period-2 cycle.

In [ ]:
F_cyc = lambda x: x**3 - 2*x + 2
dF_cyc = lambda x: 3*x**2 - 2

iterates = newton_raphson(F_cyc, dF_cyc, 0.0, n_iter=10)
print('Cycling behavior:', [f'{xi:.4f}' for xi in iterates])

### 3.4 Divergence: $F(x) = x^{1/3}$

The Newton update is $x' = x - 3x = -2x$, so $|x'| = 2|x| > |x|$ --- the iterates diverge.

In [ ]:
x = 2.0
print(f'Divergence of Newton--Raphson for F(x) = x^(1/3), starting from x0 = {x}:')
for k in range(8):
    print(f'  k={k}: x = {x:.4f}')
    x = -2 * x  # Newton update for x^(1/3)

## 4. Newton Fractals

When $F(z)$ is a polynomial with roots in $\mathbb{C}$, Newton--Raphson from different initial points $z_0 = a + bi$ converges to different roots. Coloring each $z_0$ by which root it converges to produces a **fractal** boundary between the basins of attraction.

This means **no matter how small the perturbation**, nearby initial points can converge to different roots.

**References:**
- [3Blue1Brown: Newton's Fractal](https://www.3blue1brown.com/lessons/newtons-fractal)
- [Paul Bourke: Newton--Raphson Fractals](https://paulbourke.net/fractals/newtonraphson/)
- [SciPython: Newton Fractal Code](https://scipython.com/book2/chapter-8-scipy/examples/the-newton-fractal/)

In [ ]:
TOL = 1e-8
MAX_IT = 200

def newton_complex(z0, f, fprime):
    """Newton--Raphson in the complex plane."""
    z = z0
    for _ in range(MAX_IT):
        dz = f(z) / fprime(z)
        if abs(dz) < TOL:
            return z
        z -= dz
    return False

def plot_newton_fractal(f, fprime, title='', n=500, domain=(-1.5, 1.5, -1.5, 1.5)):
    """Plot Newton fractal for f(z) = 0."""
    roots = []
    m = np.zeros((n, n))
    colors = ['#7B97AD', '#C47A6A', '#7A9A7E', '#B8995E', '#907EA0']

    def get_root_index(r):
        try:
            return np.where(np.isclose(roots, r, atol=TOL))[0][0]
        except IndexError:
            roots.append(r)
            return len(roots) - 1

    xmin, xmax, ymin, ymax = domain
    for ix, x in enumerate(np.linspace(xmin, xmax, n)):
        for iy, y in enumerate(np.linspace(ymin, ymax, n)):
            z0 = x + y * 1j
            r = newton_complex(z0, f, fprime)
            if r is not False:
                ir = get_root_index(r)
                m[iy, ix] = ir

    nroots = len(roots)
    cmap = ListedColormap(colors[:nroots]) if nroots <= len(colors) else 'hsv'
    plt.figure(figsize=(7, 7))
    plt.imshow(m, cmap=cmap, origin='lower', extent=[xmin, xmax, ymin, ymax])
    plt.xlabel('Re($z$)')
    plt.ylabel('Im($z$)')
    plt.title(title, fontsize=14)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    print(f'Found {nroots} roots: {[f"{r:.4f}" for r in roots]}')

In [ ]:
# Newton fractal for z^3 - 1 = 0 (three cube roots of unity)
plot_newton_fractal(
    f=lambda z: z**3 - 1,
    fprime=lambda z: 3*z**2,
    title='Newton Fractal: $z^3 - 1 = 0$',
    n=500
)

In [ ]:
# Newton fractal for z^4 - 1 = 0 (four fourth roots of unity)
plot_newton_fractal(
    f=lambda z: z**4 - 1,
    fprime=lambda z: 4*z**3,
    title='Newton Fractal: $z^4 - 1 = 0$',
    n=500
)

## 5. Damped Newton and Line Search

The pathologies above can be remedied by **damping** the Newton step:

$$x_{k+1} = x_k + \alpha_k \cdot d_k$$

where $d_k$ is the Newton direction and $\alpha_k$ is chosen by **backtracking line search**: find the largest $\alpha$ such that $|F(x_{k+1})| < |F(x_k)|$.

In [ ]:
def newton_backtracking(F, dF, x0, n_iter=20, beta=0.5):
    """Newton--Raphson with backtracking line search."""
    iterates = [x0]
    for _ in range(n_iter):
        d = -F(x0) / dF(x0)
        alpha = 1.0
        while abs(F(x0 + alpha * d)) >= abs(F(x0)) and alpha > 1e-10:
            alpha *= beta
        x0 += alpha * d
        iterates.append(x0)
    return iterates

def newton_constant_step(F, dF, x0, alpha=1.0, n_iter=20):
    """Newton--Raphson with constant stepsize."""
    iterates = [x0]
    for _ in range(n_iter):
        d = -F(x0) / dF(x0)
        x0 += alpha * d
        iterates.append(x0)
    return iterates

In [ ]:
# Cycling example: F(x) = x^3 - 2x + 2, x0 = 0
F_cyc = lambda x: x**3 - 2*x + 2
dF_cyc = lambda x: 3*x**2 - 2

it_vanilla = newton_constant_step(F_cyc, dF_cyc, 0.0, alpha=1.0, n_iter=20)
it_small = newton_constant_step(F_cyc, dF_cyc, 0.0, alpha=0.5, n_iter=20)
it_bt = newton_backtracking(F_cyc, dF_cyc, 0.0, n_iter=20)

print(f'{"k":>3}  {"Vanilla (alpha=1)":>18}  {"Small step (alpha=0.5)":>22}  {"Backtracking":>18}')
print('-' * 70)
for k in range(min(15, len(it_vanilla))):
    print(f'{k:3d}  {it_vanilla[k]:18.6f}  {it_small[k]:22.6f}  {it_bt[k]:18.6f}')

## 6. Convergence Experiments

We compare Newton's method with gradient descent on several convex objectives, demonstrating the two-phase convergence (Phase I: linear, Phase II: quadratic).

### 6.1 Soft-max function

$f(x_1, x_2) = e^{x_1 + 3x_2 - 0.1} + e^{x_1 - 3x_2 - 0.1} + e^{-x_1 - 0.1}$

In [ ]:
import autograd.numpy as anp
from autograd import grad, hessian

def newton_method(f, x_init, stepsize=1.0, n_iter=50):
    """Newton's method with constant stepsize."""
    grad_f = grad(f)
    hess_f = hessian(f)
    x = x_init
    values = [f(x)]
    for _ in range(n_iter):
        g = grad_f(x)
        H = hess_f(x)
        try:
            d = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            break
        x = x + stepsize * d
        values.append(f(x))
    return values

def grad_descent(f, x_init, stepsize=0.1, n_iter=100):
    """Gradient descent with constant stepsize."""
    grad_f = grad(f)
    x = x_init
    values = [f(x)]
    for _ in range(n_iter):
        x = x - stepsize * grad_f(x)
        values.append(f(x))
    return values

In [ ]:
def f_softmax(x):
    return anp.exp(x[0] + 3*x[1] - 0.1) + anp.exp(x[0] - 3*x[1] - 0.1) + anp.exp(-x[0] - 0.1)

x_init = np.array([1.0, 2.0])
vals_nt = newton_method(f_softmax, x_init, stepsize=0.3, n_iter=40)
vals_gd = grad_descent(f_softmax, x_init, stepsize=0.01, n_iter=100)

p_star = min(min(vals_nt), min(vals_gd))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(range(len(vals_nt)), [v - p_star + 1e-16 for v in vals_nt], 'o-',
             color='#7A9A7E', label='Newton', markersize=5)
ax1.semilogy(range(len(vals_gd)), [v - p_star + 1e-16 for v in vals_gd], 's-',
             color='#7B97AD', label='Gradient Descent', markersize=3)
ax1.set_xlabel('Iteration $k$')
ax1.set_ylabel('$f(x_k) - p^*$')
ax1.set_title('Suboptimality gap')
ax1.legend()

# log-log plot for Newton to show quadratic convergence
nt_gaps = [np.log(max(v - p_star, 1e-16)) for v in vals_nt[3:15]]
ax2.plot(range(3, 3 + len(nt_gaps)), [np.log(max(g, -35)) for g in nt_gaps], 'o-',
         color='#7A9A7E', markersize=6)
ax2.set_xlabel('Iteration $k$')
ax2.set_ylabel('$\\log\\log(f(x_k) - p^*)$')
ax2.set_title('Double-log plot (Newton) -- linear = quadratic convergence')

plt.tight_layout()
plt.show()

### 6.2 Logistic Regression

In [ ]:
def simulate_logistic_data(n, d, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d))
    beta_true = rng.standard_normal(d)
    probs = 1 / (1 + np.exp(-X @ beta_true))
    y = rng.binomial(1, probs)
    return X, y, beta_true

def make_logistic_loss(X, y):
    def loss(beta):
        z = X @ beta
        p = 1 / (1 + anp.exp(-z))
        return -anp.sum(y * anp.log(p) + (1 - y) * anp.log(1 - p))
    return loss

n, d = 100, 10
X, y, beta_true = simulate_logistic_data(n, d)
f_logistic = make_logistic_loss(X, y)

x_init = np.ones(d)
vals_nt = newton_method(f_logistic, x_init, stepsize=0.3, n_iter=50)
vals_gd = grad_descent(f_logistic, x_init, stepsize=0.01, n_iter=100)

p_star = min(min(vals_nt), min(vals_gd))

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(range(len(vals_nt)), [v - p_star + 1e-16 for v in vals_nt], 'o-',
            color='#7A9A7E', label='Newton', markersize=5)
ax.semilogy(range(len(vals_gd)), [v - p_star + 1e-16 for v in vals_gd], 's-',
            color='#7B97AD', label='Gradient Descent', markersize=3)
ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x_k) - p^*$')
ax.set_title('Logistic Regression: Newton vs Gradient Descent')
ax.legend()
plt.tight_layout()
plt.show()